# Result Context Extractor

## Purpose
This notebook summarizes experiment logs into publication-ready comparison tables for model configuration and group-wise generalization analysis.

## Data Sources
- `sampled_all.csv`: compact run subset used for quick configuration comparison.
- `all.csv`: full experiment log used for model and dataset-level analysis.
- `groups.csv`: merged group-level metrics for final-step ranking.

## A. Configuration Analysis on `sampled_all.csv`
This section extracts final validation metrics per run, derives configuration attributes from run names, and compares attention/embedding design choices.

**Paper usage**
- Use `main_table` for headline model ranking.
- Use `attention_mechanism_table` and `embedding_table` for ablation discussion.
- Use `feature_table` to discuss feature-wise error behavior.


### Table: Configuration-Level Final Metrics
This table lists final-step metrics for each run in `sampled_all.csv` with decoded configuration fields and paper-ready column names.


In [1]:
import pandas as pd
# Load CSV
df = pd.read_csv("sampled_all.csv")

# Make sure numeric columns are numeric
df["step"] = pd.to_numeric(df["step"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")

# Drop bad rows if any
df = df.dropna(subset=["metric", "Run", "step", "value"])

# Sort for clean lines
df = df.sort_values(["metric", "Run", "step"])

# Get final value per metric and run
final_values = (
    df.sort_values("step")
      .groupby(["metric", "Run"])
      .last()
      .reset_index()
)

# Get last step number per run
last_steps = (
    df.groupby("Run")["step"]
      .max()
      .reset_index(name="last_step")
)

# Pivot final metric values
table = (
    final_values.pivot(
        index="Run",
        columns="metric",
        values="value"
    )
    .reset_index()
)

table.drop(columns=["val_mbe"], inplace=True, errors="ignore")
table.columns.name = None

# Add last step into table
table = table.merge(last_steps, on="Run", how="left")

# derive features from Run name
# extract attention and embedding status directly from Run
table["attention"] = table["Run"].str.extract(r'(yes_attention|no_attention)', expand=False)
table["embedding"] = table["Run"].str.extract(r'(yes_embedding|no_embedding)', expand=False)

table["attention_type"] = (
    table["Run"]
    .str.extract(r"(bahdanau|mlp)", expand=False)
    .fillna("none")
)

# keep only yes / no
table["attention"] = table["attention"].str.replace("_attention", "", regex=False)
table["embedding"] = table["embedding"].str.replace("_embedding", "", regex=False)

# rename Run → run_id
table = table.rename(columns={"Run": "run_id"})

table = table.sort_values(by="attention_type")

# create readable run labels
table.insert(0, "Run", [f"{i+1}" for i in range(len(table))])
table.drop(columns=["run_id"], inplace=True)

# reorder columns so derived columns come after run_id
cols = ["Run", "attention_type", "attention", "embedding", "last_step"] + [
    c for c in table.columns
    if c not in ["Run", "attention", "embedding", "attention_type", "last_step"]
]

table = table[cols]
table

paper_col_map = {
    "Run": "Run Number",
    "run_id": "Run Name",
    "config": "Configuration",
    "machine_part": "Machine Part",
    "model": "Model",
    "attention_type": "Attention Mechanism",
    "attention": "Feature Separated Attention",
    "embedding": "Embedding Enabled",
    "group_number": "Group ID",
    "step": "Final Step",
    "last_step": "Last Training Step",
    "train_loss": "Training Loss",
    "val_loss": "Validation Loss",
    "val_mae": "Validation MAE",
    "val_medae": "Validation MedAE",
    "val_mse": "Validation MSE",
    "val_rmse": "Validation RMSE",
    "val_r2": "Validation R2",
    "val_mse_feature_0": "Validation MSE (Feature 0)",
    "val_mse_feature_1": "Validation MSE (Feature 1)",
    "val_r2_feature_0": "Validation R2 (Feature 0)",
    "val_r2_feature_1": "Validation R2 (Feature 1)"
}

table.rename(columns=paper_col_map).T


,0,1,3,6,2,4,5,7
Run Number,1,2,3,4,5,6,7,8
Attention Mechanism,bahdanau,bahdanau,bahdanau,bahdanau,mlp,mlp,mlp,mlp
Feature Separated Attention,yes,no,no,yes,no,no,yes,yes
Embedding Enabled,no,no,yes,yes,no,yes,no,yes
Last Training Step,28,115,93,98,91,122,105,116
Training Loss,0.271748,0.108548,0.101151,0.101219,0.097196,0.099838,0.123343,0.109133
Validation Loss,0.183831,0.156962,0.128611,0.131754,0.148276,0.135291,0.143359,0.15664
Validation MAE,0.066108,0.052446,0.048578,0.050097,0.050939,0.049118,0.053694,0.052163
Validation MedAE,0.050013,0.040165,0.037676,0.042504,0.038942,0.037778,0.04273,0.040091
Validation MSE,0.008834,0.004942,0.004344,0.004215,0.004722,0.004439,0.005257,0.005079


### Table: Full Validation Summary (`sampled_all`)
This table reports the main validation metrics (`Validation Loss`, `Validation MAE`, `Validation MedAE`, `Validation MSE`, `Validation RMSE`, `Validation R2`) per run.


In [2]:
full_table = table[[
    "Run","attention_type","attention","embedding","last_step",
    "val_loss","val_mae","val_medae","val_mse","val_rmse","val_r2"
]]

full_table = full_table.rename(columns=paper_col_map)
full_table.T


,0,1,3,6,2,4,5,7
Run Number,1,2,3,4,5,6,7,8
Attention Mechanism,bahdanau,bahdanau,bahdanau,bahdanau,mlp,mlp,mlp,mlp
Feature Separated Attention,yes,no,no,yes,no,no,yes,yes
Embedding Enabled,no,no,yes,yes,no,yes,no,yes
Last Training Step,28,115,93,98,91,122,105,116
Validation Loss,0.183831,0.156962,0.128611,0.131754,0.148276,0.135291,0.143359,0.15664
Validation MAE,0.066108,0.052446,0.048578,0.050097,0.050939,0.049118,0.053694,0.052163
Validation MedAE,0.050013,0.040165,0.037676,0.042504,0.038942,0.037778,0.04273,0.040091
Validation MSE,0.008834,0.004942,0.004344,0.004215,0.004722,0.004439,0.005257,0.005079
Validation RMSE,0.093992,0.070301,0.065909,0.064921,0.068714,0.066628,0.072504,0.071266


### Table: Main Ranking by RMSE (`sampled_all`)
This table ranks configurations by `Validation RMSE` (ascending) for primary model-selection comparison.


In [3]:
main_table = table[[
    "attention_type","attention","embedding",
    "val_rmse","val_mae","val_r2",
]].sort_values("val_rmse")
main_table = main_table.rename(columns=paper_col_map)
main_table.T


,6,3,4,2,1,7,5,0
Attention Mechanism,bahdanau,bahdanau,mlp,mlp,bahdanau,mlp,mlp,bahdanau
Feature Separated Attention,yes,no,no,no,no,yes,yes,yes
Embedding Enabled,yes,yes,yes,no,no,yes,no,no
Validation RMSE,0.064921,0.065909,0.066628,0.068714,0.070301,0.071266,0.072504,0.093992
Validation MAE,0.050097,0.048578,0.049118,0.050939,0.052446,0.052163,0.053694,0.066108
Validation R2,0.790505,0.801102,0.787804,0.771814,0.749695,0.757339,0.778359,0.048844


### Table: Attention Mechanism Ablation (`sampled_all`)
This table shows mean validation performance grouped by attention mechanism type.


In [4]:
attention_mechanism_table = (
    table.groupby("attention_type")[["val_rmse","val_mae","val_r2"]]
    .mean()
    .reset_index()
)

attention_mechanism_table = attention_mechanism_table.rename(columns=paper_col_map)
attention_mechanism_table


,Attention Mechanism,Validation RMSE,Validation MAE,Validation R2
0,bahdanau,0.073781,0.054307,0.597536
1,mlp,0.069778,0.051479,0.773829


### Table: Embedding Ablation (`sampled_all`)
This table shows mean validation performance grouped by embedding usage.


In [5]:
embedding_table = (
    table.groupby("embedding")[["val_rmse","val_mae","val_r2"]]
    .mean()
    .reset_index()
)

embedding_table = embedding_table.rename(columns=paper_col_map)
embedding_table


,Embedding Enabled,Validation RMSE,Validation MAE,Validation R2
0,no,0.076378,0.055797,0.587178
1,yes,0.067181,0.049989,0.784187


### Table: Feature-Wise Error Breakdown (`sampled_all`)
This table compares total and per-feature validation MSE across configurations.


In [6]:
feature_table = table[[
    "attention_type","attention","embedding",
    "val_mse", "val_mse_feature_0","val_mse_feature_1",
    "val_r2", "val_r2_feature_0","val_r2_feature_1"
]].sort_values("val_mse")
feature_table = feature_table.rename(columns=paper_col_map)
feature_table.T


,6,3,4,2,1,7,5,0
Attention Mechanism,bahdanau,bahdanau,mlp,mlp,bahdanau,mlp,mlp,bahdanau
Feature Separated Attention,yes,no,no,no,no,yes,yes,yes
Embedding Enabled,yes,yes,yes,no,no,yes,no,no
Validation MSE,0.004215,0.004344,0.004439,0.004722,0.004942,0.005079,0.005257,0.008834
Validation MSE (Feature 0),0.002816,0.002536,0.002784,0.003014,0.003402,0.003182,0.002658,0.003151
Validation MSE (Feature 1),0.005614,0.006152,0.006095,0.006429,0.006483,0.006975,0.007856,0.014518
Validation R2,0.790505,0.801102,0.787804,0.771814,0.749695,0.757339,0.778359,0.048844
Validation R2 (Feature 0),0.685882,0.717138,0.689469,0.663724,0.620489,0.64498,0.703467,0.124499
Validation R2 (Feature 1),0.895129,0.885067,0.886139,0.879903,0.8789,0.869698,0.853252,0.056289


## B. Full Run Analysis on `all.csv`
This section scales the same extraction logic to all runs and adds `model` and `machine_part` factors for cross-dataset/model comparisons.

**Paper usage**
- Use `main_results_df` as the complete result table.
- Use `best_per_group_df` to report best configuration per (`machine_part`, `model`).
- Use `model_comparison_df` and `dataset_comparison_df` for aggregate performance statements.


### Table: Full-Run Final Metrics (`all.csv`)
This table summarizes final-step metrics for all runs with machine part, model, and architecture options.


In [7]:
import pandas as pd
import matplotlib.pyplot as plt

# Load CSV
df = pd.read_csv("all.csv")

# Make sure numeric columns are numeric
df["step"] = pd.to_numeric(df["step"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")

# Drop bad rows if any
df = df.dropna(subset=["metric", "Run", "step", "value"])

# Sort for clean lines
df = df.sort_values(["metric", "Run", "step"])

# Get final value per metric and run
final_values = (
    df.sort_values("step")
      .groupby(["metric", "Run"])
      .last()
      .reset_index()
)

# Get last step number per run
last_steps = (
    df.groupby("Run")["step"]
      .max()
      .reset_index(name="last_step")
)

# Pivot final metric values
table = (
    final_values.pivot(
        index="Run",
        columns="metric",
        values="value"
    )
    .reset_index()
)

table.drop(columns=["val_mbe"], inplace=True, errors="ignore")
table.columns.name = None

# Add last step into table
table = table.merge(last_steps, on="Run", how="left")

# derive features from Run name
# extract attention and embedding status directly from Run
table["attention"] = table["Run"].str.extract(r"(yes_attention|no_attention)", expand=False)
table["embedding"] = table["Run"].str.extract(r"(yes_embedding|no_embedding)", expand=False)

table["attention_type"] = (
    table["Run"]
    .str.extract(r"(bahdanau|mlp)", expand=False)
    .fillna("none")
)

# extract model
# first match tcn_lstm explicitly, otherwise match lstm only if tcn_lstm is not present
table["model"] = table["Run"].str.extract(r"(tcn_lstm)", expand=False).fillna("lstm")
#table["model"]= table["Run"].str.extract(r"\b(lstm)\b", expand=False)

# extract machine part
table["machine_part"] = table["Run"].str.extract(r"^(All|Bending)", expand=False)

# keep only yes / no
table["attention"] = table["attention"].str.replace("_attention", "", regex=False)
table["embedding"] = table["embedding"].str.replace("_embedding", "", regex=False)

# fill missing values if needed
table["attention"] = table["attention"].fillna("none")
table["embedding"] = table["embedding"].fillna("none")
table["model"] = table["model"].fillna("unknown")
table["machine_part"] = table["machine_part"].fillna("unknown")
table["attention_type"] = table["attention_type"].fillna("none")

# rename Run → run_id
table = table.rename(columns={"Run": "run_id"})

table = table.sort_values(by=["machine_part", "model", "attention_type"])

# create readable run labels
table.insert(0, "Run", [f"{i+1}" for i in range(len(table))])

# reorder columns
cols = ["Run", "machine_part", "model", "attention_type", "attention", "embedding", "last_step"] + [
    c for c in table.columns
    if c not in ["Run", "machine_part", "run_id", "model", "attention", "embedding", "attention_type", "last_step"]
]

table = table[cols]

table

paper_col_map = {
    "Run": "Run Number",
    "run_id": "Run Name",
    "config": "Configuration",
    "machine_part": "Machine Part",
    "model": "Model",
    "attention_type": "Attention Mechanism",
    "attention": "Feature Separated Attention",
    "embedding": "Embedding Enabled",
    "group_number": "Group ID",
    "step": "Final Step",
    "last_step": "Last Training Step",
    "train_loss": "Training Loss",
    "val_loss": "Validation Loss",
    "val_mae": "Validation MAE",
    "val_medae": "Validation MedAE",
    "val_mse": "Validation MSE",
    "val_rmse": "Validation RMSE",
    "val_r2": "Validation R2",
    "val_mse_feature_0": "Validation MSE (Feature 0)",
    "val_mse_feature_1": "Validation MSE (Feature 1)",
    "val_r2_feature_0": "Validation R2 (Feature 0)",
    "val_r2_feature_1": "Validation R2 (Feature 1)"
}

table.rename(columns=paper_col_map)


,Run Number,Machine Part,Model,Attention Mechanism,Feature Separated Attention,Embedding Enabled,Last Training Step,Training Loss,Validation Loss,Validation MAE,Validation MedAE,Validation MSE,Validation MSE (Feature 0),Validation MSE (Feature 1),Validation R2,Validation R2 (Feature 0),Validation R2 (Feature 1),Validation RMSE
0,1,All,tcn_lstm,bahdanau,no,no,83,0.140515,0.179865,0.060373,0.047922,0.006363,0.003483,0.009243,0.719377,0.611420,0.827334,0.079769
1,2,All,tcn_lstm,bahdanau,no,yes,83,0.083507,0.126406,0.046421,0.036542,0.003903,0.002831,0.004975,0.795629,0.684191,0.907068,0.062473
2,3,All,tcn_lstm,bahdanau,yes,no,54,0.123849,0.173686,0.059288,0.048198,0.005994,0.003568,0.008421,0.722349,0.602003,0.842695,0.077422
3,4,All,tcn_lstm,bahdanau,yes,yes,65,0.093802,0.169707,0.056592,0.046424,0.005534,0.003469,0.007598,0.735524,0.612983,0.858066,0.074388
4,5,All,tcn_lstm,none,no,no,97,0.146418,0.194684,0.058564,0.046163,0.006251,0.004052,0.008449,0.695063,0.547965,0.842161,0.079061
5,6,All,tcn_lstm,none,no,yes,44,0.101744,0.163332,0.053402,0.042193,0.005107,0.003424,0.006789,0.745604,0.618039,0.873169,0.071461
6,7,All,tcn_lstm,none,yes,no,73,0.122160,0.162473,0.057430,0.046270,0.005913,0.002910,0.008916,0.754421,0.675393,0.833450,0.076894
7,8,All,tcn_lstm,none,yes,yes,63,0.099653,0.135836,0.049394,0.040090,0.004261,0.003000,0.005522,0.781104,0.665365,0.896844,0.065275
8,9,Bending,lstm,bahdanau,no,no,115,0.079992,0.150890,0.054520,0.042473,0.005372,0.002855,0.007889,0.767061,0.681487,0.852636,0.073293
9,10,Bending,lstm,bahdanau,no,yes,90,0.074515,0.114650,0.045550,0.036788,0.003610,0.002663,0.004558,0.808894,0.702931,0.914858,0.060086


### Table: Main Results by Dataset and Model
This table reports per-run core metrics grouped by machine part and model family.


In [8]:
main_results_df = table[[
    "machine_part",
    "model",
    "attention_type",
    "attention",
    "embedding",
    "val_rmse",
    "val_mae",
    "val_r2"
]].sort_values(["machine_part","model","val_rmse"])

main_results_df = main_results_df.rename(columns=paper_col_map)
main_results_df


,Machine Part,Model,Attention Mechanism,Feature Separated Attention,Embedding Enabled,Validation RMSE,Validation MAE,Validation R2
1,All,tcn_lstm,bahdanau,no,yes,0.062473,0.046421,0.795629
7,All,tcn_lstm,none,yes,yes,0.065275,0.049394,0.781104
5,All,tcn_lstm,none,no,yes,0.071461,0.053402,0.745604
3,All,tcn_lstm,bahdanau,yes,yes,0.074388,0.056592,0.735524
6,All,tcn_lstm,none,yes,no,0.076894,0.057430,0.754421
2,All,tcn_lstm,bahdanau,yes,no,0.077422,0.059288,0.722349
4,All,tcn_lstm,none,no,no,0.079061,0.058564,0.695063
0,All,tcn_lstm,bahdanau,no,no,0.079769,0.060373,0.719377
15,Bending,lstm,none,yes,yes,0.058029,0.043407,0.823285
11,Bending,lstm,bahdanau,yes,yes,0.058526,0.044211,0.820370


### Table: Best Configuration per (`Machine Part`, `Model`)
This table selects the run with minimum `Validation RMSE` within each machine-part/model subgroup.


In [9]:
best_per_group_df = (
    table.loc[
        table.groupby(["machine_part","model"])["val_rmse"].idxmin()
    ][[
        "machine_part",
        "model",
        "attention_type",
        "attention",
        "embedding",
        "val_rmse",
        "val_mae",
        "val_r2"
    ]]
)

best_per_group_df = best_per_group_df.rename(columns=paper_col_map)
best_per_group_df


,Machine Part,Model,Attention Mechanism,Feature Separated Attention,Embedding Enabled,Validation RMSE,Validation MAE,Validation R2
1,All,tcn_lstm,bahdanau,no,yes,0.062473,0.046421,0.795629
15,Bending,lstm,none,yes,yes,0.058029,0.043407,0.823285
23,Bending,tcn_lstm,none,yes,yes,0.061779,0.046518,0.816842


### Table: Average Effect of Embedding (`all.csv`)
This table reports mean validation metrics aggregated by embedding usage.


In [10]:
embedding_effect_df = (
    table.groupby("embedding")[["val_rmse","val_mae","val_r2"]]
    .mean()
    .reset_index()
)

embedding_effect_df = embedding_effect_df.rename(columns=paper_col_map)
embedding_effect_df


,Embedding Enabled,Validation RMSE,Validation MAE,Validation R2
0,no,0.072248,0.054463,0.764325
1,yes,0.063847,0.048041,0.794523


### Table: Average Effect of Attention Mechanism (`all.csv`)
This table reports mean validation metrics aggregated by attention mechanism type.


In [11]:
attention_mechanism_df = (
    table.groupby("attention_type")[["val_rmse","val_mae","val_r2"]]
    .mean()
    .reset_index()
)

attention_mechanism_df = attention_mechanism_df.rename(columns=paper_col_map)
attention_mechanism_df


,Attention Mechanism,Validation RMSE,Validation MAE,Validation R2
0,bahdanau,0.068274,0.051572,0.778235
1,none,0.067820,0.050933,0.780613


### Table: Model Comparison by Machine Part
This table reports average validation metrics for each (`Machine Part`, `Model`) pair.


In [12]:
model_comparison_df = (
    table.groupby(["machine_part","model"])[["val_rmse","val_mae","val_r2"]]
    .mean()
    .reset_index()
)

model_comparison_df = model_comparison_df.rename(columns=paper_col_map)
model_comparison_df


,Machine Part,Model,Validation RMSE,Validation MAE,Validation R2
0,All,tcn_lstm,0.073343,0.055183,0.743634
1,Bending,lstm,0.063283,0.047506,0.804299
2,Bending,tcn_lstm,0.067516,0.051067,0.790339


### Table: Dataset-Level Comparison
This table reports average validation metrics for each machine part across all model/configuration variants.


In [13]:
dataset_comparison_df = (
    table.groupby("machine_part")[["val_rmse","val_mae","val_r2"]]
    .mean()
    .reset_index()
)

dataset_comparison_df = dataset_comparison_df.rename(columns=paper_col_map)
dataset_comparison_df


,Machine Part,Validation RMSE,Validation MAE,Validation R2
0,All,0.073343,0.055183,0.743634
1,Bending,0.065400,0.049287,0.797319


### Table: Feature-Wise Metrics (`all.csv`)
This table reports total and feature-level MSE/R2 for each run to analyze output-dimension behavior.


In [14]:
feature_table = table[[
    "machine_part",
    "model",
    "attention_type",
    "attention",
    "embedding",
    "val_mse",
    "val_mse_feature_0",
    "val_mse_feature_1",
    "val_r2",
    "val_r2_feature_0",
    "val_r2_feature_1"
]].sort_values("val_mse")

feature_table = feature_table.rename(columns=paper_col_map)
feature_table


,Machine Part,Model,Attention Mechanism,Feature Separated Attention,Embedding Enabled,Validation MSE,Validation MSE (Feature 0),Validation MSE (Feature 1),Validation R2,Validation R2 (Feature 0),Validation R2 (Feature 1)
15,Bending,lstm,none,yes,yes,0.003367,0.002451,0.004284,0.823285,0.726601,0.919970
11,Bending,lstm,bahdanau,yes,yes,0.003425,0.002490,0.004360,0.820370,0.722194,0.918545
9,Bending,lstm,bahdanau,no,yes,0.003610,0.002663,0.004558,0.808894,0.702931,0.914858
10,Bending,lstm,bahdanau,yes,no,0.003768,0.002392,0.005145,0.818543,0.733198,0.903888
23,Bending,tcn_lstm,none,yes,yes,0.003817,0.002409,0.005225,0.816842,0.731285,0.902399
13,Bending,lstm,none,no,yes,0.003838,0.002618,0.005058,0.806705,0.707889,0.905520
12,Bending,lstm,none,no,no,0.003858,0.002620,0.005095,0.806244,0.707669,0.904820
1,All,tcn_lstm,bahdanau,no,yes,0.003903,0.002831,0.004975,0.795629,0.684191,0.907068
17,Bending,tcn_lstm,bahdanau,no,yes,0.004027,0.002697,0.005357,0.799549,0.699169,0.899929
19,Bending,tcn_lstm,bahdanau,yes,yes,0.004142,0.002698,0.005586,0.797356,0.699061,0.895650


## C. Final-Step Group Ranking (`groups.csv`)
This section extracts the final step of each group and creates ranking tables for overall validation quality, feature-level metrics, and train-vs-val generalization.

**Paper usage**
- `val_table`: overall validation ranking.
- `feature_table`: feature-wise MSE/R2 breakdown.
- `generalization_table`: potential overfitting indicators via train/validation gap.


### Table: Final Step per Group (`groups.csv`)
This table keeps the final logged step for each group and reports consolidated metrics with paper-ready labels.


In [15]:
import pandas as pd

# -----------------------------
# 1) Read all GP metric files
# -----------------------------
df = pd.read_csv("groups.csv")

# -----------------------------
# 2) Clean numeric columns
# -----------------------------
df["step"] = pd.to_numeric(df["step"], errors="coerce")
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")

df["value"] = (
    df["value"]
    .astype(str)
    .str.replace("'", "", regex=False)
    .str.strip()
)

df["value"] = pd.to_numeric(df["value"], errors="coerce")

# -----------------------------
# 3) Extract group number
# -----------------------------
group_number = pd.to_numeric(df["Run"].str.extract(r'gp(\d+)')[0], errors="coerce")

# insert as first column
df.insert(0, "group_number", group_number)

# remove Run column
df = df.drop(columns=["Run", "Run ID"])
# each metric becomes a separate column
# pivot metrics to columns
table = df.pivot_table(
    index=["group_number", "step", "timestamp"],
    columns="metric",
    values="value",
    aggfunc="first"
).reset_index()

table.columns.name = None

# keep only the last step per group
result = table.loc[table.groupby("group_number")["step"].idxmax()].reset_index(drop=True)
result.drop(columns = ["timestamp", "val_mbe"], inplace = True)
result

paper_col_map = {
    "Run": "Run Number",
    "run_id": "Run Name",
    "config": "Configuration",
    "machine_part": "Machine Part",
    "model": "Model",
    "attention_type": "Attention Mechanism",
    "attention": "Feature Separated Attention",
    "embedding": "Embedding Enabled",
    "group_number": "Group ID",
    "step": "Final Step",
    "last_step": "Last Training Step",
    "train_loss": "Training Loss",
    "val_loss": "Validation Loss",
    "val_mae": "Validation MAE",
    "val_medae": "Validation MedAE",
    "val_mse": "Validation MSE",
    "val_rmse": "Validation RMSE",
    "val_r2": "Validation R2",
    "val_mse_feature_0": "Validation MSE (Feature 0)",
    "val_mse_feature_1": "Validation MSE (Feature 1)",
    "val_r2_feature_0": "Validation R2 (Feature 0)",
    "val_r2_feature_1": "Validation R2 (Feature 1)"
}

result.rename(columns=paper_col_map)


,Group ID,Final Step,Training Loss,Validation Loss,Validation MAE,Validation MedAE,Validation MSE,Validation MSE (Feature 0),Validation MSE (Feature 1),Validation R2,Validation R2 (Feature 0),Validation R2 (Feature 1),Validation RMSE
0,1,46,0.177250,0.252940,0.081711,0.059532,0.011895,0.003605,0.020184,0.499729,0.518415,0.481043,0.109062
1,2,56,0.109340,0.113452,0.045482,0.035902,0.003514,0.002527,0.004502,0.828473,0.734443,0.922504,0.059283
2,3,63,0.094028,0.190503,0.054389,0.044632,0.004966,0.004032,0.005900,0.636613,0.389144,0.884083,0.070470
3,4,71,0.098489,0.153492,0.055732,0.041024,0.006051,0.002139,0.009962,0.787691,0.757541,0.817842,0.077785
4,5,42,0.102484,0.456238,0.095986,0.067318,0.019734,0.003638,0.035831,0.531282,0.600069,0.462496,0.140479
5,6,16,0.345314,1.322300,0.195018,0.150090,0.063631,0.018053,0.109208,-3.536221,-4.786224,-2.286226,0.252251
6,7,16,0.689473,0.573474,0.109523,0.083156,0.022558,0.008081,0.037035,-0.705512,-0.517362,-0.893662,0.150195
7,8,25,0.174210,0.259333,0.083682,0.065025,0.012016,0.003693,0.020339,-0.232386,-0.026266,-0.438507,0.109618
8,9,16,0.344781,2.002543,0.252159,0.181874,0.091636,0.018895,0.164376,-4.091936,-3.256050,-4.927830,0.302714


In [16]:
df

,group_number,metric,step,timestamp,value
0,9,val_r2_feature_1,1,1770208037891,-3.660289
1,9,val_r2_feature_1,2,1770208063272,-4.112537
2,9,val_r2_feature_1,3,1770208088431,-4.605062
3,9,val_r2_feature_1,4,1770208113936,-4.847022
4,9,val_r2_feature_1,5,1770208139540,-4.804154
...,...,...,...,...,...
4207,1,train_loss,42,1770103577996,0.185172
4208,1,train_loss,43,1770103600407,0.185147
4209,1,train_loss,44,1770103622559,0.178802
4210,1,train_loss,45,1770103644701,0.178702


### Table: Group Ranking by Validation R2
This table ranks groups by `Validation R2` and includes complementary validation error metrics.


In [17]:
val_table = result[[
    "group_number",
    "step",
    "val_loss",
    "val_mae",
    "val_medae",
    "val_mse",
    "val_rmse",
    "val_r2"
]].sort_values("val_r2", ascending=False)

val_table = val_table.rename(columns=paper_col_map)
val_table


,Group ID,Final Step,Validation Loss,Validation MAE,Validation MedAE,Validation MSE,Validation RMSE,Validation R2
1,2,56,0.113452,0.045482,0.035902,0.003514,0.059283,0.828473
3,4,71,0.153492,0.055732,0.041024,0.006051,0.077785,0.787691
2,3,63,0.190503,0.054389,0.044632,0.004966,0.070470,0.636613
4,5,42,0.456238,0.095986,0.067318,0.019734,0.140479,0.531282
0,1,46,0.252940,0.081711,0.059532,0.011895,0.109062,0.499729
7,8,25,0.259333,0.083682,0.065025,0.012016,0.109618,-0.232386
6,7,16,0.573474,0.109523,0.083156,0.022558,0.150195,-0.705512
5,6,16,1.322300,0.195018,0.150090,0.063631,0.252251,-3.536221
8,9,16,2.002543,0.252159,0.181874,0.091636,0.302714,-4.091936


### Table: Group Feature-Wise Performance
This table ranks groups by total `Validation R2` and reports feature-level MSE/R2 values.


In [18]:
feature_table = result[[
    "group_number",
    "val_mse",
    "val_mse_feature_0",
    "val_mse_feature_1",
    "val_r2",
    "val_r2_feature_0",
    "val_r2_feature_1"
]].sort_values("val_r2", ascending=False)

feature_table = feature_table.rename(columns=paper_col_map)
feature_table


,Group ID,Validation MSE,Validation MSE (Feature 0),Validation MSE (Feature 1),Validation R2,Validation R2 (Feature 0),Validation R2 (Feature 1)
1,2,0.003514,0.002527,0.004502,0.828473,0.734443,0.922504
3,4,0.006051,0.002139,0.009962,0.787691,0.757541,0.817842
2,3,0.004966,0.004032,0.005900,0.636613,0.389144,0.884083
4,5,0.019734,0.003638,0.035831,0.531282,0.600069,0.462496
0,1,0.011895,0.003605,0.020184,0.499729,0.518415,0.481043
7,8,0.012016,0.003693,0.020339,-0.232386,-0.026266,-0.438507
6,7,0.022558,0.008081,0.037035,-0.705512,-0.517362,-0.893662
5,6,0.063631,0.018053,0.109208,-3.536221,-4.786224,-2.286226
8,9,0.091636,0.018895,0.164376,-4.091936,-3.256050,-4.927830


### Table: Generalization Summary by Group
This table compares `Training Loss`, `Validation Loss`, and `Validation R2` per group to inspect generalization behavior.


In [19]:
generalization_table = result[[
    "group_number",
    "train_loss",
    "val_loss",
    "val_r2"
]].sort_values("val_r2", ascending=False)

generalization_table = generalization_table.rename(columns=paper_col_map)
generalization_table


,Group ID,Training Loss,Validation Loss,Validation R2
1,2,0.109340,0.113452,0.828473
3,4,0.098489,0.153492,0.787691
2,3,0.094028,0.190503,0.636613
4,5,0.102484,0.456238,0.531282
0,1,0.177250,0.252940,0.499729
7,8,0.174210,0.259333,-0.232386
6,7,0.689473,0.573474,-0.705512
5,6,0.345314,1.322300,-3.536221
8,9,0.344781,2.002543,-4.091936


## E. Interpretation Checklist for Writing
Use the generated tables to explicitly answer:
1. Which configuration minimizes `val_rmse` overall and per subset?
2. Does attention type improve both error (`RMSE`, `MAE`) and explained variance (`R2`)?
3. Is embedding consistently beneficial across `machine_part` and model families?
4. Are there groups with strong train loss but weak validation (`generalization_table`) indicating overfitting?
